# Food Recipe Provenance Classification

**Italian vs American recipes | Machine Learning Final Project**

This project predicts recipe cuisine from ingredient-based TF-IDF features, using supervised feature engineering, model selection, and weighted ensemble learning.

**Dataset:** 3,200 labelled recipes (`1 = American`, `2 = Italian`), 1,734 unlabelled recipes, and 40 ingredient TF-IDF features.

**Approach:** Add four cuisine-profile features within cross-validation pipelines, tune an RBF SVM, Random Forest and ExtraTrees, and combine five models using weighted soft voting.


## 1. Setup and imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

RANDOM_STATE = 42

# Prefer a repository-style data/ folder, but also support local CSV files.
DATA_DIR = Path("data") if Path("data/train.csv").exists() else Path(".")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load data

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

TARGET_COL = "y"
feature_cols = [c for c in train_df.columns if c != TARGET_COL]
X = train_df[feature_cols].copy()
y = train_df[TARGET_COL].astype(int).copy()
X_test = test_df[feature_cols].copy()

print(f"Training: {X.shape}; held-out: {X_test.shape}")
print("Class counts:", y.value_counts().sort_index().to_dict())

## 3. Evaluation metric and cross-validation

I minimize the proportion of misclassified recipes (0–1 loss). The custom scorer returns the negative loss so that `GridSearchCV` can maximize its score. Model evaluation uses shuffled, stratified **10-fold cross-validation** (`random_state=42`).

In [ ]:
W = {(1, 1): 0.0, (1, 2): 1.0, (2, 1): 1.0, (2, 2): 0.0}

def weighted_loss(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return float(np.mean([W[(int(p), int(t))] for p, t in zip(y_pred, y_true)]))

def weighted_score(y_true, y_pred):
    return -weighted_loss(y_true, y_pred)

weighted_scorer = make_scorer(weighted_score, response_method="predict")
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

## 4. Leakage-safe feature engineering: `CuisineProfileAugmenter`

Adds **four** features to the original 40 TF-IDF columns: cosine similarity to each class centroid, the difference between those similarities, and the weighted ingredient log-ratio score. Since the supervised augmenter is **inside each sklearn pipeline**, class profiles are estimated only from the training fold during CV.

In [ ]:
class CuisineProfileAugmenter(BaseEstimator, TransformerMixin):
    def __init__(self, add_original=True, eps=1e-6):
        self.add_original = add_original
        self.eps = eps

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        if not np.all(np.isin(y, [1, 2])):
            raise ValueError("expects labels 1 and 2")
        self.mean_american_ = X[y == 1].mean(axis=0)
        self.mean_italian_ = X[y == 2].mean(axis=0)
        self.norm_american_ = np.linalg.norm(self.mean_american_) + self.eps
        self.norm_italian_ = np.linalg.norm(self.mean_italian_) + self.eps
        self.log_ratio_ = np.log(
            (self.mean_italian_ + self.eps) / (self.mean_american_ + self.eps)
        )
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        x_norm = np.linalg.norm(X, axis=1) + self.eps
        cos_am = (X @ self.mean_american_) / (x_norm * self.norm_american_)
        cos_it = (X @ self.mean_italian_) / (x_norm * self.norm_italian_)
        profile = np.column_stack([
            cos_am, cos_it, cos_it - cos_am, X @ self.log_ratio_
        ])
        return np.column_stack([X, profile]) if self.add_original else profile

## 5. Model preprocessing pipelines

For SVM: imputation → supervised profile augmentation → scaling → classifier. For tree models: imputation → supervised profile augmentation → classifier.

In [ ]:
def scaled_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("profile", CuisineProfileAugmenter(add_original=True)),
        ("scaler", StandardScaler()),
        ("clf", model),
    ])

def tree_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("profile", CuisineProfileAugmenter(add_original=True)),
        ("clf", model),
    ])

## 6. Hyperparameter tuning

Tune the RBF SVM, Random Forest, and ExtraTrees classifiers separately using stratified 10-fold cross-validation. The best hyperparameters and corresponding validation losses are obtained from the grid searches below.

In [ ]:
svm_search = GridSearchCV(
    estimator=scaled_pipeline(SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
    param_grid={
        "clf__C": [3.0, 4.0, 5.0, 6.0, 8.0],
        "clf__gamma": [0.05, 0.06, 0.075, 0.1],
    },
    scoring=weighted_scorer, cv=cv10, n_jobs=1, verbose=2,
)
svm_search.fit(X, y)
print("SVM best:", svm_search.best_params_, "CV loss:", -svm_search.best_score_)

In [ ]:
rf_search = GridSearchCV(
    estimator=tree_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    param_grid={
        "clf__n_estimators": [300, 500],
        "clf__max_features": ["sqrt", 0.5],
        "clf__min_samples_leaf": [1, 2, 3],
        "clf__max_depth": [None, 12, 20],
    },
    scoring=weighted_scorer, cv=cv10, n_jobs=1, verbose=2,
)
rf_search.fit(X, y)
print("RF best:", rf_search.best_params_, "CV loss:", -rf_search.best_score_)

In [ ]:
et_search = GridSearchCV(
    estimator=tree_pipeline(ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    param_grid={
        "clf__n_estimators": [500, 800],
        "clf__max_features": ["sqrt", 0.4, 0.5, 0.6],
        "clf__min_samples_leaf": [1, 2],
        "clf__max_depth": [None, 20],
    },
    scoring=weighted_scorer, cv=cv10, n_jobs=1, verbose=2,
)
et_search.fit(X, y)
print("ExtraTrees best:", et_search.best_params_, "CV loss:", -et_search.best_score_)

## 7. Seed-bagging soft-voting ensemble

Combine **one RBF SVM, two Random Forest models, and two ExtraTrees models**. Each tree-model pair uses different random seeds to introduce diversity into the ensemble.

In [ ]:
def strip_prefix(params, prefix="clf__"):
    return {k.replace(prefix, "", 1): v for k, v in params.items()}

best_svm = strip_prefix(svm_search.best_params_)
best_rf = strip_prefix(rf_search.best_params_)
best_et = strip_prefix(et_search.best_params_)

def make_svm_component():
    return scaled_pipeline(SVC(kernel="rbf", probability=True,
                               random_state=RANDOM_STATE, **best_svm))

def make_rf_component(seed):
    return tree_pipeline(RandomForestClassifier(random_state=seed, n_jobs=-1, **best_rf))

def make_et_component(seed):
    return tree_pipeline(ExtraTreesClassifier(random_state=seed, n_jobs=-1, **best_et))

In [ ]:
class SimpleSoftVotingClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimators, weights=None):
        self.estimators = estimators
        self.weights = weights

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.fitted_estimators_ = []
        for name, est in self.estimators:
            self.fitted_estimators_.append((name, clone(est).fit(X, y)))
        return self

    def predict_proba(self, X):
        w = self.weights if self.weights is not None else np.ones(len(self.fitted_estimators_))
        s = sum(wi * est.predict_proba(X)
                for wi, (_, est) in zip(w, self.fitted_estimators_))
        return s / np.sum(w)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

def make_model(weights):
    return SimpleSoftVotingClassifier(
        estimators=[
            ("svm", make_svm_component()),
            ("rf_seed_101", make_rf_component(101)),
            ("rf_seed_202", make_rf_component(202)),
            ("et_seed_404", make_et_component(303)),
            ("et_seed_505", make_et_component(404)),
        ],
        weights=weights,
    )

## 8. Ensemble weight selection

Compare 16 candidate weight vectors using 10-fold cross-validation and select the configuration with the lowest mean validation loss.

In [ ]:
weight_grid = [
    [7, 0, 0, 2, 2], [8, 0, 0, 2, 2], [9, 0, 0, 2, 2],
    [7, .25, .25, 2, 2], [8, .25, .25, 2, 2], [9, .25, .25, 2, 2],
    [7, .5, .5, 2, 2], [8, .5, .5, 2, 2], [9, .5, .5, 2, 2],
    [7, .5, .5, 3, 3], [8, .5, .5, 3, 3], [9, .5, .5, 3, 3],
    [8, 0, 0, 3, 3], [9, 0, 0, 3, 3],
    [8, .25, .25, 3, 3], [9, .25, .25, 3, 3],
]

weight_results = []
for w in weight_grid:
    scores = cross_validate(make_model(w), X, y, scoring=weighted_scorer,
                            cv=cv10, n_jobs=1)
    weight_results.append({"weights": w, "cv_loss": -scores["test_score"].mean()})

weight_results = pd.DataFrame(weight_results).sort_values("cv_loss")
best_weights = weight_results.iloc[0]["weights"]
print("Best weights:", best_weights)
display(weight_results.head(10))

## 9. Fit final model and export held-out predictions

In [ ]:
final_model = make_model(best_weights)
final_model.fit(X, y)

test_pred = final_model.predict(X_test)
np.savetxt(OUTPUT_DIR / "seed_42_submission.txt", test_pred, fmt="%s")
print("Saved. Class counts:", dict(zip(*np.unique(test_pred, return_counts=True))))

## 10. Ingredient interpretation

Explore differences in mean ingredient TF-IDF profiles across cuisines and identify the most discriminative features using importance scores from the fitted ExtraTrees component.

In [ ]:
import matplotlib.patches as mpatches

aug = CuisineProfileAugmenter(add_original=False)
aug.fit(X.values, y.values)
log_ratio = pd.Series(aug.log_ratio_, index=feature_cols)

top_italian = log_ratio.nlargest(10)
top_american = log_ratio.nsmallest(10)
signature = pd.concat([top_italian, top_american])

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#C0392B" if v > 0 else "#2980B9" for v in signature.values]
ax.barh(signature.index[::-1], signature.values[::-1], color=colors[::-1])
ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.5)
ax.set_xlabel("Log-ratio (positive = Italian, negative = American)")
ax.set_title("Top ingredients characterising each cuisine")
it_patch = mpatches.Patch(color="#C0392B", label="Italian")
am_patch = mpatches.Patch(color="#2980B9", label="American")
ax.legend(handles=[it_patch, am_patch])
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_log_ratio.png", dpi=150)
plt.show()

In [ ]:
et_clf = final_model.fitted_estimators_[3][1].named_steps["clf"]
importances = pd.Series(et_clf.feature_importances_[:40], index=feature_cols).nlargest(15)
bar_colors = ["#C0392B" if log_ratio[i] > 0 else "#2980B9"
              for i in importances.index]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(importances.index[::-1], importances.values[::-1], color=bar_colors[::-1])
ax.set_xlabel("Mean decrease in impurity (ExtraTrees)")
ax.set_title("Top 15 most discriminative ingredients")
ax.legend(handles=[it_patch, am_patch])
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_et_importance.png", dpi=150)
plt.show()